# RViT+ v5 — train on a Colab GPU

Runs the conv-free **memory-as-tokens** RViT+ v5 agent on a cloud GPU using your
Colab credits, reading the code straight from your existing Google Drive link.

**Before you run anything:** *Runtime → Change runtime type → Hardware accelerator = GPU.*
The free tier gives a **T4** (no credits used). To spend **compute units**, you need
Colab Pro / Pro+ or pay-as-you-go, then pick a premium GPU (**L4** or **A100**) in that
same dialog. Long runs that survive closing the tab need **Pro+** (background execution).


In [ ]:
!nvidia-smi -L || echo "No GPU detected -> Runtime > Change runtime type > GPU"

### 1. Mount your Drive (the same Drive this project is synced to)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Locate the AttentionManuscript folder on Drive
It may live under **My Drive** *or* under **Othercomputers** (if it's a backed-up
computer folder rather than a My-Drive folder). This cell checks both without a slow
full-Drive crawl. If it prints `None`, read the listed paths and set `REPO` by hand.

In [ ]:
import os

def find_repo():
    cands = ['/content/drive/MyDrive/AttentionManuscript',
             '/content/drive/My Drive/AttentionManuscript']
    oc = '/content/drive/Othercomputers'
    if os.path.isdir(oc):
        for dev in sorted(os.listdir(oc)):
            d = os.path.join(oc, dev)
            cands.append(os.path.join(d, 'AttentionManuscript'))
            if os.path.isdir(d):
                for sub in sorted(os.listdir(d)):
                    cands.append(os.path.join(d, sub, 'AttentionManuscript'))
    # a repo is valid only if it actually contains the v5 package
    return next((c for c in cands if os.path.isdir(os.path.join(c, 'RViT_plus_v5'))), None)

REPO = find_repo()
if REPO:
    os.environ['REPO'] = REPO
    print('Found repo at:', REPO)
else:
    print('Could not auto-locate it. Inspect these and set REPO manually:')
    print(' /content/drive ->', os.listdir('/content/drive'))
    for r in ('/content/drive/MyDrive', '/content/drive/Othercomputers'):
        if os.path.isdir(r):
            print(f' {r} ->', os.listdir(r)[:25])
    # REPO = '/content/drive/.../AttentionManuscript'
    # os.environ['REPO'] = REPO

### 3. Install the one missing dependency (Colab already has torch + numpy)

In [ ]:
!pip install -q gymnasium
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

### 4. Quick smoke test on the GPU (optional but recommended)
Confirms the model builds and one update runs before you commit to a long run.

In [ ]:
!cd "$REPO" && python -m RViT_plus_v5.tests.test_v5

### 5. Train

**Checkpoints are written to Colab-local `/content`, NOT to Drive.** Your config
already keeps checkpoints outside the Drive-synced repo because Drive rewriting a
file mid-run can corrupt it; the Colab Drive (FUSE) mount makes frequent small
writes worse. We copy them to Drive afterward (next cell).

The env rollout is single-env and CPU/numpy-bound, so the GPU mainly accelerates the
PPO **update** (re-encoding the B×T batch). Larger `--episodes-per-iter` / `--per-n-replay`
give the GPU a bigger update batch and better utilization.

In [ ]:
!cd "$REPO" && python RViT_plus_v5/train_rl.py --device cuda --iters 2000 --episodes-per-iter 8 --per-n-replay 4 --checkpoint-dir /content/rvit_v5_ckpts --save-every 100 --log-every 5

### 6. Persist checkpoints back to Drive (so they survive the runtime)

In [ ]:
import os, glob, shutil
dst = os.path.join(os.environ['REPO'], 'RViT_plus_v5', 'checkpoints')
os.makedirs(dst, exist_ok=True)
for pt in sorted(glob.glob('/content/rvit_v5_ckpts/*.pt')):
    shutil.copy2(pt, dst)
    print('copied', os.path.basename(pt), '->', dst)

### 7. If the runtime disconnects: resume
Re-run cells 1-3, copy the last Drive checkpoint back to `/content`, then resume.
Note: resume reloads **model weights only** — the iteration counter, optimizer state,
and PER replay buffer restart fresh (the trainer checkpoints weights, not full state).

```python
!cp "$REPO"/RViT_plus_v5/checkpoints/rvit_plus_v5_rl_latest.pt /content/rvit_v5_ckpts/ 2>/dev/null
!cd "$REPO" && python RViT_plus_v5/train_rl.py --device cuda --init-mode resume \
    --checkpoint-path /content/rvit_v5_ckpts/rvit_plus_v5_rl_latest.pt \
    --checkpoint-dir /content/rvit_v5_ckpts --iters 2000 --save-every 100
```

_(To run v4 or v3 instead, swap `RViT_plus_v5` for `RViT_plus_v4` / `RViT_plus_v3`.)_